# Train on SageMaker Studio

Goal:

- Train a YOLO model to detect license plates.
- Run in a SageMaker Studio JupyterLab notebook.
- Read raw data from `s3://<bucket>/data/raw/`.
- Export the trained model to `s3://<bucket>/trains/models/`.


## Environment

Inspect the runtime before doing anything else.

The Studio image ships PyTorch but not `ultralytics`, so install it first.
**Restart the kernel after the install cell**, then run the rest of the notebook —
torch is already imported by the image and a new version will not take effect otherwise.


In [ ]:
# The Studio image has no ultralytics, and ships a mismatched torch/torchvision
# pair. Let pip resolve them together.
%pip install -q -U ultralytics torch torchvision onnx onnxslim

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import torch
import torchvision
import ultralytics

# SageMaker Studio ships SDK v3, where Session lives under sagemaker.core.
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Repo root: the notebook lives in <repo>/jupyter-notebook
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Local working copies on the space EBS volume. S3 is the source of truth.
RAW = ROOT / "data" / "raw"              # synced from s3://<bucket>/data/raw/
PROCESSED = ROOT / "data" / "processed"  # split, synced to s3://<bucket>/data/split/
RUNS = ROOT / "runs"                     # ultralytics training output
MODELS = ROOT / "models"                 # exported model -> s3://<bucket>/trains/models/

for d in (RAW, PROCESSED, RUNS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

session = Session()
REGION = session.boto_region_name
ROLE = get_execution_role()

# Bucket name carries a random terraform suffix, so the notebook-init lifecycle
# script writes it to this env file on app start (see infra/scripts/notebook-init.sh).
env_file = Path.home() / ".sagemaker-yolo.env"
if "BUCKET" not in os.environ and env_file.exists():
    for line in env_file.read_text().splitlines():
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip())

BUCKET = os.environ["BUCKET"]  # restart the app if this is missing

S3_RAW = f"s3://{BUCKET}/data/raw"
S3_SPLIT = f"s3://{BUCKET}/data/split"
S3_MODELS = f"s3://{BUCKET}/trains/models"

# Use the GPU if the space has one, otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("torchvision ", torchvision.__version__)
print("ultralytics ", ultralytics.__version__)
print("sagemaker-core ", version("sagemaker-core"))
print("cuda        ", torch.cuda.is_available())
print("device      ", DEVICE)
print("region      ", REGION)
print("bucket      ", BUCKET)
print("root        ", ROOT)

## Data processing

`data/` is gitignored, so the cloned repo has no images. Pull the raw dataset
down from S3 into the space, inspect it, split it, then push the split back.


In [ ]:
import subprocess

# Pull raw data into the space. Idempotent: only new/changed objects transfer.
print(f"{S3_RAW}/  ->  {RAW}")
subprocess.run(["aws", "s3", "sync", S3_RAW, str(RAW), "--only-show-errors"], check=True)

n_images = len([p for p in RAW.iterdir() if p.suffix.lower() in {".jpeg", ".jpg", ".png"}])
n_labels = len([p for p in RAW.iterdir() if p.suffix.lower() == ".txt" and p.name != "classes.txt"])
print(f"images {n_images}  labels {n_labels}")

if n_images == 0:
    raise RuntimeError(
        f"no images under {S3_RAW}/ — upload the dataset first:\n"
        f"  aws s3 sync data/raw {S3_RAW}"
    )

In [ ]:
# Inspect the raw dataset before splitting: pairing, box counts, malformed labels.
from src.data_loader import summarize

stats = summarize(RAW)
for key, value in stats.items():
    print(f"{key:22} {value}")

Visualize labels with images.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from src.data_loader import find_pairs


def show_sample(pairs, n=6, seed=0):
    """Draw label boxes on a few images to confirm the annotations line up."""
    import random

    sample = random.Random(seed).sample(pairs, n)
    fig, axes = plt.subplots(2, n // 2, figsize=(14, 6))
    for ax, (img_path, lbl_path) in zip(axes.ravel(), sample):
        img = Image.open(img_path)
        ax.imshow(img)
        w, h = img.size
        for line in lbl_path.read_text().splitlines():
            if not line.strip():
                continue
            _, cx, cy, bw, bh = (float(v) for v in line.split())
            # YOLO stores normalised centre + size; matplotlib wants corners.
            ax.add_patch(
                plt.Rectangle(
                    ((cx - bw / 2) * w, (cy - bh / 2) * h),
                    bw * w,
                    bh * h,
                    fill=False,
                    edgecolor="lime",
                    linewidth=2,
                )
            )
        ax.set_title(img_path.stem[:28], fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


pairs, _, _ = find_pairs(RAW)
show_sample(pairs)

### Split

80/20 train/val, seeded so the split is reproducible.
`LIMIT` caps the number of pairs for a fast smoke run.


In [ ]:
from src.data_loader import build_split, verify_split

# limit for a fast smoke run; e.g. 200
LIMIT = None  # unlimited
# LIMIT = 200

# build_split wipes PROCESSED and re-copies from RAW, so it is safe to re-run
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=0))

# fails loudly on unpaired files or train/val leakage
print(verify_split(PROCESSED))

Push the split to S3 so the exact train/val partition is recoverable later.
`--delete` keeps the prefix in step with a re-run under a different seed or limit.


In [ ]:
# Push the split back so the exact partition is recoverable later.
print(f"{PROCESSED}  ->  {S3_SPLIT}/")
subprocess.run(
    ["aws", "s3", "sync", str(PROCESSED), S3_SPLIT, "--delete", "--only-show-errors"],
    check=True,
)

# confirm object counts landed, without listing every key
listing = subprocess.run(
    ["aws", "s3", "ls", f"{S3_SPLIT}/", "--recursive", "--summarize"],
    check=True, capture_output=True, text=True,
)
print("\n".join(ln for ln in listing.stdout.splitlines() if "Total" in ln))

### Dataset configuration file

Write `configs/data.yaml`, which points YOLO at the split and names the classes.
Class names come from `classes.txt` in the raw data.


In [ ]:
from src.data_loader import write_data_yaml

# classes.txt ships with the raw data; one class name per line
names = (RAW / "classes.txt").read_text().split()

# write_data_yaml creates configs/ and writes an absolute `path`,
# so training works regardless of cwd
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)

print(data_yaml)
print(data_yaml.read_text())

## Define model

Load the pretrained checkpoint and confirm it lines up with the dataset.

- `yolo11n` is the smallest YOLO11 variant, which matters on a CPU-only space.


### Training hyperparameters

Generated by `build_train_cfg()` rather than read from a checked-in file, so
`device` and `workers` follow the detected runtime. Any key can be overridden
by keyword, which is what the hyperparameter sweep in stage 5 does per trial.

Ultralytics writes the fully resolved config to `runs/<name>/args.yaml`, so the
run's exact settings are recorded without keeping a copy here.


In [ ]:
from src.data_loader import build_train_cfg

# device/workers come from the runtime detected in the environment section;
# override any other key here, e.g. build_train_cfg(..., epochs=50, batch=16)
train_cfg = build_train_cfg(
    device=DEVICE,
    workers=os.cpu_count() or 2,
)

# `project` is relative in the config; anchor it to the repo, not the cwd
train_cfg["project"] = str(ROOT / train_cfg["project"])

train_cfg

In [ ]:
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

# Fail here rather than several minutes into training.
checked = check_det_dataset(str(data_yaml))
print("dataset ", {k: checked[k] for k in ("nc", "names", "train", "val")})

# Pretrained checkpoint named by the config; downloaded on first use.
model = YOLO(train_cfg["model"])

n_params = sum(p.numel() for p in model.model.parameters())
print(f"model    {train_cfg['model']}  {n_params / 1e6:.2f}M params")

# The head is built for COCO's 80 classes; ultralytics reshapes it to `nc`
# at train time, so a mismatch here is expected.
print(f"classes  pretrained {len(model.names)} -> dataset {checked['nc']} {list(checked['names'].values())}")

## Train

Train YOLO model.


In [ ]:
import time

cfg = dict(train_cfg)
model_weights = cfg.pop("model")  # `model` is not a model.train() kwarg

# construct model class with parameters
model = YOLO(model_weights)

# log start time
start = time.time()

# run the training process, get performance results
results = model.train(data=str(data_yaml), **cfg)

# print elapsed time
print(f"\nelapsed: {time.time() - start:.0f}s")

Print performance results


In [ ]:
# print performance results dir
save_dir = Path(results.save_dir)
print(save_dir)

# Final-epoch metrics on the validation split.
for key in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    print(f"{key:24} {results.results_dict[key]:.4f}")

print()
# list model file with weights
for weight in sorted((save_dir / "weights").glob("*.pt")):
    print(f"{weight.name:10} {weight.stat().st_size / 1e6:.1f} MB")

## Validate the model

Built-in method used to validate and evaluate the performance of a trained model


In [ ]:
# read model from path
best = YOLO(str(save_dir / "weights" / "best.pt"))
IMGSZ = 640


# evaluate the performance of a trained model
metrics = best.val(data=str(data_yaml), imgsz=IMGSZ, batch=8, device="cpu", plots=False)

print(f"mAP50      {metrics.box.map50:.4f}")
print(f"mAP50-95   {metrics.box.map:.4f}")
print(f"precision  {metrics.box.mp:.4f}")
print(f"recall     {metrics.box.mr:.4f}")

### Predictions vs fact

- Green = label
- Red = prediction


In [ ]:
import random


def show_predictions(model, split_dir, n=6, seed=0, conf=0.25):
    """Overlay ground-truth (green) and predicted (red) boxes on val images."""
    images = sorted((split_dir / "images").iterdir())
    sample = random.Random(seed).sample(images, n)

    fig, axes = plt.subplots(2, n // 2, figsize=(15, 7))
    for ax, img_path in zip(axes.ravel(), sample):
        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)

        label_path = split_dir / "labels" / f"{img_path.stem}.txt"
        n_true = 0
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            n_true += 1
            _, cx, cy, bw, bh = (float(v) for v in line.split())
            ax.add_patch(
                plt.Rectangle(
                    ((cx - bw / 2) * w, (cy - bh / 2) * h), bw * w, bh * h,
                    fill=False, edgecolor="lime", linewidth=2,
                )
            )

        result = model.predict(img_path, imgsz=416, conf=conf, device="cpu", verbose=False)[0]
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            ax.add_patch(
                plt.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    fill=False, edgecolor="red", linewidth=2, linestyle="--",
                )
            )
            ax.text(x1, y1 - 4, f"{box.conf.item():.2f}", color="red", fontsize=8)

        ax.set_title(f"true {n_true} / pred {len(result.boxes)}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_predictions(best, PROCESSED / "val")

### Missing predictions

List missing predictions


In [ ]:
from collections import defaultdict

val_dir = PROCESSED / "val"
by_true_count = defaultdict(lambda: {"images": 0, "true": 0, "pred": 0, "under": 0})

for img_path in sorted((val_dir / "images").iterdir()):
    label_path = val_dir / "labels" / f"{img_path.stem}.txt"
    n_true = len([ln for ln in label_path.read_text().splitlines() if ln.strip()])
    n_pred = len(
        best.predict(img_path, imgsz=IMGSZ, conf=0.25, device="cpu", verbose=False)[0].boxes
    )

    row = by_true_count[n_true]
    row["images"] += 1
    row["true"] += n_true
    row["pred"] += n_pred
    row["under"] += max(0, n_true - n_pred)

print(f"{'plates/img':>11} {'images':>7} {'labelled':>9} {'detected':>9} {'missed':>7}")
for n_true in sorted(by_true_count):
    r = by_true_count[n_true]
    print(f"{n_true:>11} {r['images']:>7} {r['true']:>9} {r['pred']:>9} {r['under']:>7}")

total_true = sum(r["true"] for r in by_true_count.values())
total_missed = sum(r["under"] for r in by_true_count.values())
print(f"\nmissed {total_missed}/{total_true} plates ({total_missed / total_true:.1%})")

## Export to ONNX

Export the trained model for serving.


In [ ]:
import shutil

# imgsz must match training
exported = Path(best.export(format="onnx", imgsz=train_cfg["imgsz"], opset=12, simplify=True))

# count the split used to train the model
n_images = sum(len(list((PROCESSED / s / "images").iterdir())) for s in ("train", "val"))

# <name>-<images>img-<epochs>ep-<imgsz>px.onnx
onnx_path = MODELS / (
    f"{train_cfg['name']}-{n_images}img-{train_cfg['epochs']}ep-{train_cfg['imgsz']}px.onnx"
)
shutil.move(str(exported), onnx_path)

print(onnx_path)
print(f"{onnx_path.stat().st_size / 1e6:.1f} MB")

Export metadata.


In [ ]:
import json

sidecar = onnx_path.with_suffix(".metadata.json")
sidecar.write_text(json.dumps({
    "imgsz": train_cfg["imgsz"],
    "names": [best.names[i] for i in sorted(best.names)],
}, indent=2))

print(sidecar.read_text())